# Week 4: Dashboard Preparation

This notebook prepares data for Google Looker Studio dashboard.

## Dashboard Requirements

1. **KPI Scorecards**:
   - Total Sales
   - Total Profit
   - Order Count
   - Average Order Value

2. **Visualizations**:
   - Sales trend over time (line chart)
   - Sales by category (bar chart)
   - Profit by region (bar chart)
   - Sales by segment (pie chart)

3. **Interactive Filters**:
   - Date range
   - Region
   - Category
   - Segment

In [ ]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv('../week-03-python-analysis/data/superstore_cleaned.csv', 
                 parse_dates=['Order Date', 'Ship Date'])

print(f"Loaded {len(df):,} rows")
print(f"Date range: {df['Order Date'].min()} to {df['Order Date'].max()}")

## Calculate KPIs

In [ ]:
# Calculate KPIs
kpis = {
    'Total Sales': df['Sales'].sum(),
    'Total Profit': df['Profit'].sum(),
    'Order Count': df['Order ID'].nunique(),
    'Average Order Value': df['Sales'].sum() / df['Order ID'].nunique(),
    'Profit Margin': (df['Profit'].sum() / df['Sales'].sum()) * 100,
    'Unique Customers': df['Customer ID'].nunique()
}

print("="*50)
print("KEY PERFORMANCE INDICATORS")
print("="*50)
for kpi, value in kpis.items():
    if 'Sales' in kpi or 'Profit' in kpi or 'Value' in kpi:
        print(f"{kpi}: ${value:,.2f}")
    elif 'Margin' in kpi:
        print(f"{kpi}: {value:.1f}%")
    else:
        print(f"{kpi}: {value:,}")

## Prepare Data for Export

In [ ]:
# Create aggregated datasets for dashboard

# 1. Monthly summary
monthly = df.groupby([df['Order Date'].dt.to_period('M').astype(str)]).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique',
    'Customer ID': 'nunique'
}).reset_index()
monthly.columns = ['Month', 'Sales', 'Profit', 'Orders', 'Customers']
monthly.to_csv('../data/monthly_summary.csv', index=False)
print(f"✓ Saved monthly_summary.csv ({len(monthly)} rows)")

# 2. By Category
category = df.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique'
}).reset_index()
category.to_csv('../data/by_category.csv', index=False)
print(f"✓ Saved by_category.csv ({len(category)} rows)")

# 3. By Region
region = df.groupby('Region').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'nunique'
}).reset_index()
region.to_csv('../data/by_region.csv', index=False)
print(f"✓ Saved by_region.csv ({len(region)} rows)")

# 4. Full dataset for detailed filtering
df_export = df[[
    'Order Date', 'Category', 'Sub-Category', 'Region', 'Segment',
    'Sales', 'Profit', 'Quantity', 'Discount'
]].copy()
df_export.to_csv('../data/dashboard_data.csv', index=False)
print(f"✓ Saved dashboard_data.csv ({len(df_export)} rows)")

## Dashboard Configuration Guide

### Steps to Create Dashboard in Looker Studio:

1. **Connect Data**:
   - Go to [Looker Studio](https://datastudio.google.com)
   - Create new report
   - Upload `dashboard_data.csv`

2. **Add Scorecards**:
   - Add chart → Scorecard
   - Metric: Sales (Sum)
   - Name: "Total Sales"
   - Repeat for Profit, Orders

3. **Add Charts**:
   - Line chart: Month vs Sales
   - Bar chart: Category vs Sales
   - Bar chart: Region vs Profit
   - Pie chart: Segment breakdown

4. **Add Filters**:
   - Date range control
   - Drop-down: Region
   - Drop-down: Category
   - Drop-down: Segment

5. **Publish**:
   - Click Share → Get report link
   - Set to "Anyone with link can view"
   - Copy link for submission